# RAG-проект, КТ5: генерация, оценка качества и продвинутый RAG

Последняя КТ. Генерирую ответы на весь золотой набор по выдаче из КТ3 и КТ4 и оцениваю их LLM-судьёй. Потом делаю агентский RAG, проверяю судью руками и в конце свожу всё по проекту в одну таблицу.

In [1]:
%pip install -q chromadb rank-bm25 sentence-transformers scikit-learn pandas matplotlib "openai==2.54.0"

In [2]:
import warnings
warnings.filterwarnings('ignore')
import hashlib, json, os, re, time, pickle, random, threading
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_VERBOSITY'] = 'error'
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import chromadb
from openai import OpenAI
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import cohen_kappa_score
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()
import logging
logging.getLogger('sentence_transformers').setLevel(logging.ERROR)

%matplotlib inline
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 80)

try:
    from google.colab import userdata
    os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
except ImportError:
    pass

DATA = Path('data') if Path('data').exists() else Path('../data')
KT4 = DATA / 'kt4_cache'
CACHE = DATA / 'kt5_cache'
(CACHE / 'llm').mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

gold = json.load(open(DATA / 'gold_set.json', encoding='utf-8'))
val_q = [g for g in gold if g['split'] == 'val']
test_q = [g for g in gold if g['split'] == 'test']

def load_pkl(name):
    p = CACHE / name
    return pickle.load(open(p if p.exists() else KT4 / name, 'rb'))

LOCK = threading.RLock()
def save_pkl(obj, name):
    with LOCK:
        pickle.dump(obj, open(CACHE / f'{name}.tmp', 'wb'))
        os.replace(CACHE / f'{name}.tmp', CACHE / name)

TIMES = load_pkl('timings.pkl')
def measured(key, fn):
    if key not in TIMES:
        TIMES[key] = fn()
        save_pkl(TIMES, 'timings.pkl')
    return TIMES[key]
print('val:', len(val_q), '| test:', len(test_q))

val: 35 | test: 15


## 0. Данные

Корпус, чанки и метрики поиска те же, что в КТ3 и КТ4.

In [3]:
docs_clean, dup_map = pickle.load(open(DATA / 'kt2_cache' / 'clean_corpus.pkl', 'rb'))
chunks, _ = pickle.load(open(DATA / 'kt2_cache' / 'chunks_sentence_128.pkl', 'rb'))
STOP = set(open(DATA / 'stop_ru.txt', encoding='utf-8').read().split())
doc_text = {d['doc_id']: d['text'] for d in docs_clean}

def content_type(t):
    t = t.lower()
    if re.search(r'розыгрыш|победител|конкурс', t): return 'contest'
    if re.search(r'реклама|erid|igm\.gg|промокод', t): return 'ad'
    if re.search(r'получила скидку|покупаем за|акция продлится|топ \d+ игр до|\| -\d+% \||-\d+%\s*\|', t): return 'deal'
    if re.search(r'раздач|бесплатно|забрать', t): return 'giveaway'
    return 'news'

tf = TfidfVectorizer(max_features=30000, min_df=2, max_df=0.3, token_pattern=r'(?u)\b[а-яёa-z][а-яёa-z0-9]{2,}\b', stop_words=list(STOP))
M = tf.fit_transform([c['text'].lower() for c in chunks])
vocab = np.array(tf.get_feature_names_out())
SRC = {'egs_tg': 'Epic Games Store', 'SteamByFree': 'Халявный Steam'}
CT = {'news': 'новость', 'deal': 'скидка', 'giveaway': 'раздача', 'contest': 'розыгрыш', 'ad': 'реклама'}
texts = []
for i, c in enumerate(chunks):
    row = M.getrow(i)
    keywords = ', '.join(vocab[row.indices[np.argsort(-row.data)][:5]])
    texts.append(f"Канал: {SRC[c['source']]}. Дата: {str(c['date'])[:10]}. Тип: {CT[content_type(doc_text[c['doc_id']])]}. Ключевые слова: {keywords}.\n{c['text']}")
chunk_doc = np.array([c['doc_id'] for c in chunks])
print('чанков:', len(texts))

чанков: 2938


In [4]:
KS = (1, 5, 10)
MAIN = ['P@5', 'R@5', 'NDCG@5']

def top_posts(idx, n=10):
    out = []
    for i in idx:
        if chunk_doc[i] not in out:
            out.append(chunk_doc[i])
    return out[:n]

def evaluate(search, qs):
    rows = []
    for g in qs:
        rel = {dup_map.get(d, d) for d in g['relevant_doc_ids']}
        if not rel:
            continue
        hits = [p in rel for p in top_posts(search(g['question']))]
        r = {'qid': g['qid'], 'type': g['type']}
        for k in KS:
            r[f'P@{k}'] = sum(hits[:k]) / k
            r[f'R@{k}'] = sum(hits[:k]) / len(rel)
            ideal = sum(1 / np.log2(i + 2) for i in range(min(len(rel), k)))
            r[f'NDCG@{k}'] = sum(1 / np.log2(i + 2) for i, h in enumerate(hits[:k]) if h) / ideal
        r['MRR'] = next((1 / (i + 1) for i, h in enumerate(hits) if h), 0.0)
        rows.append(r)
    return pd.DataFrame(rows)

def summary(df, cols=MAIN):
    return df[cols].mean().round(3)

## 1. Baseline: поиск из КТ3 и КТ4

Поднимаю все конфигурации поиска из КТ3 и КТ4. Выдачу Chroma и ответы LLM-реранкера беру из кэша КТ4: реранкер там работал на Claude, и для этих вопросов его ответы уже сохранены. Для генерации нужны две конфигурации: baseline retrieval это hybrid RRF из КТ3, best retrieval это hybrid с LLM-реранкером из КТ4. Остальные пригодятся для сводной таблицы в конце.

In [5]:
WORD = re.compile(r'\w+')
bm25_tok = lambda t: WORD.findall(t.lower()) or ['_']
bm25 = BM25Okapi([bm25_tok(t) for t in texts], k1=0.6, b=1.0)

frida = SentenceTransformer('ai-forever/FRIDA', device=DEVICE)
frida.max_seq_length = 512
emb_path = DATA / 'kt3_cache' / 'emb' / 'FRIDA_docs.npy'
E = np.load(emb_path) if emb_path.exists() else frida.encode(texts, prompt='search_document: ', batch_size=32, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
collection = chromadb.PersistentClient(path=str(KT4 / 'chroma')).get_or_create_collection('posts', embedding_function=None, configuration={'hnsw': {
    'space': 'cosine', 'max_neighbors': 16, 'ef_construction': 100, 'ef_search': 100}})
if collection.count() == 0:
    for i in range(0, len(E), 1000):
        collection.add(ids=[str(j) for j in range(i, min(i + 1000, len(E)))], embeddings=E[i:i + 1000])
DEPTH = 50

QEMB, DENSE = load_pkl('query_emb.pkl'), load_pkl('dense_results.pkl')
def embed_queries(qs):
    with LOCK:
        new = [q for q in dict.fromkeys(qs) if q not in QEMB]
        if not new:
            return
        for q, v in zip(new, frida.encode(new, prompt='search_query: ', batch_size=32, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)):
            QEMB[q] = v
        save_pkl(QEMB, 'query_emb.pkl')

def chroma_query(q, n):
    res = collection.query(query_embeddings=[QEMB[q]], n_results=n, include=['distances'])
    return [int(i) for i in res['ids'][0]]

def dense_list(q, n=DEPTH):
    with LOCK:
        if (q, n) not in DENSE:
            embed_queries([q])
            DENSE[q, n] = chroma_query(q, n)
            save_pkl(DENSE, 'dense_results.pkl')
        return DENSE[q, n]

def bm25_list(q, n=DEPTH):
    return list(np.argsort(-bm25.get_scores(bm25_tok(q)))[:n])

def rrf(lists, n, k=60):
    score = {}
    for lst in lists:
        for rank, i in enumerate(lst):
            score[i] = score.get(i, 0) + 1 / (k + rank + 1)
    return sorted(score, key=score.get, reverse=True)[:n]

def hybrid(q, n=DEPTH):
    return rrf([bm25_list(q, n), dense_list(q, n)], n)

def kt4_llm(kind, prompt, run=0):
    key = hashlib.md5(f'{prompt}|{run}'.encode()).hexdigest()
    return json.loads((KT4 / 'llm' / f'{kind}__{key}.json').read_text(encoding='utf-8'))

MQ_PROMPT = '''Пользователь ищет информацию в Telegram-каналах про игры: раздачи, скидки, новости игровой индустрии. Перепиши его вопрос тремя разными способами, чтобы по ним было удобно искать посты. Исправь опечатки, разговорные слова замени обычными, названия игр и компаний пиши так, как их обычно пишут. Если в вопросе спрашивают про несколько вещей, раздели его на отдельные вопросы. Верни только 3 варианта, каждый с новой строки, без нумерации и пояснений.

Вопрос: {q}'''

def mq_variants(q):
    o = kt4_llm('mq', MQ_PROMPT.replace('{q}', q))
    lines = [re.sub(r'^\s*(?:\d+[.)]|[-•*])\s*', '', l).strip() for l in o['text'].split('\n')]
    return [l for l in lines if l][:3]

def multi_query(q, n=DEPTH):
    lists = []
    for v in [q] + mq_variants(q):
        lists += [bm25_list(v, n), dense_list(v, n)]
    return rrf(lists, n)

ce = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', device=DEVICE, max_length=512)
CE_POOL = 20
def ce_hybrid(q):
    cand = hybrid(q)
    scores = ce.predict([(q, texts[i]) for i in cand[:CE_POOL]], batch_size=32, show_progress_bar=False)
    return [cand[j] for j in np.argsort(-scores)] + cand[CE_POOL:]

LR_N = 10
LR_V3 = 'Ниже вопрос пользователя и {n} фрагментов постов из Telegram-каналов про игры. Оцени каждый фрагмент: 2 если он прямо отвечает на вопрос, 1 если он по теме и отвечает частично, 0 если не помогает. Верни только оценки всех {n} фрагментов в формате номер:оценка через запятую, например: 1:2, 2:0, 3:1. Без пояснений.\n\nВопрос: {q}\n\nФрагменты:\n{docs}'
LR_FORMAT = re.compile(r'\s*\d+\s*:\s*[012](?:\s*,\s*\d+\s*:\s*[012])*\s*\.?\s*')

def lr_call(q, cand):
    docs = '\n\n'.join(f'[{j + 1}] ' + texts[i].replace('\n', ' ') for j, i in enumerate(cand))
    prompt = LR_V3.replace('{n}', str(len(cand))).replace('{q}', q).replace('{docs}', docs)
    o = kt4_llm('lr', prompt)
    if LR_FORMAT.fullmatch(o['text']) and sorted(int(j) for j in re.findall(r'(\d+)\s*:', o['text'])) == list(range(1, len(cand) + 1)):
        return o
    retry = kt4_llm('lr', prompt + f'\n\nОтвет должен быть только списком из {len(cand)} оценок номер:оценка через запятую, без текста.')
    return {'text': retry['text'], 'in': o['in'] + retry['in'], 'out': o['out'] + retry['out'], 'sec': o['sec'] + retry['sec']}

def lr_hybrid(q):
    cand = hybrid(q)
    score = {int(j) - 1: int(v) for j, v in re.findall(r'(\d+)\s*:\s*([012])', lr_call(q, cand[:LR_N])['text']) if 0 < int(j) <= LR_N}
    return [cand[j] for j in sorted(range(LR_N), key=lambda j: (-score.get(j, 0), j))] + cand[LR_N:]

RETRIEVERS = {
    'КТ3: BM25': bm25_list,
    'КТ3: dense FRIDA': dense_list,
    'КТ3: hybrid RRF': hybrid,
    'КТ4: + multi-query': multi_query,
    'КТ4: + cross-encoder': ce_hybrid,
    'КТ4: + LLM-реранкер': lr_hybrid,
}
TOP5 = {name: {g['qid']: list(map(int, r(g['question'])[:5])) for g in gold} for name, r in RETRIEVERS.items()}
RET = {name: (evaluate(r, val_q), evaluate(r, test_q)) for name, r in RETRIEVERS.items()}
pd.DataFrame([{'конфигурация': name, **{f'val {m}': v for m, v in summary(a).items()}, **{f'test {m}': v for m, v in summary(b).items()}}
              for name, (a, b) in RET.items()])

,конфигурация,val P@5,val R@5,val NDCG@5,test P@5,test R@5,test NDCG@5
0,КТ3: BM25,0.236,0.737,0.719,0.250,0.750,0.655
1,КТ3: dense FRIDA,0.214,0.699,0.675,0.233,0.729,0.730
2,КТ3: hybrid RRF,0.236,0.743,0.713,0.250,0.771,0.759
3,КТ4: + multi-query,0.229,0.698,0.655,0.250,0.771,0.709
4,КТ4: + cross-encoder,0.229,0.729,0.719,0.267,0.792,0.681
5,КТ4: + LLM-реранкер,0.250,0.763,0.734,0.267,0.792,0.761


Цифры совпали с КТ3 и КТ4, значит всё поднялось так же.

В КТ5 генерация и судья работают на DeepSeek. Ответы пишет deepseek-flash, по документации это версия DeepSeek-V4.1-Flash-1017. Судья deepseek-v4-pro, версия DeepSeek-V4-Pro-0813. Судья специально другая и более сильная модель, чтобы модель не оценивала сама себя. У обеих по умолчанию включено размышление, а с ним temperature не работает, поэтому размышление выключаю и ставлю temperature 0. Модели и промпт генерации дальше по КТ5 не меняются. Все ответы сохраняются в data/kt5_cache, при перезапуске API не нужен. Цену считаю по дневному тарифу DeepSeek, это верхняя оценка.

In [6]:
GEN_MODEL, JUDGE_MODEL = 'deepseek-flash', 'deepseek-v4-pro'
PRICE = {'deepseek-flash': (0.3 / 1e6, 1.2 / 1e6), 'deepseek-v4-pro': (1.32 / 1e6, 3.96 / 1e6), 'claude-sonnet-4-6': (3 / 1e6, 15 / 1e6)}
_client = None

def chat(kind, messages, max_tokens, model=GEN_MODEL, run=0, tools=None):
    global _client
    key = hashlib.md5(json.dumps([model, messages, tools, run], ensure_ascii=False).encode()).hexdigest()
    p = CACHE / 'llm' / f'{kind}__{key}.json'
    if p.exists():
        return json.loads(p.read_text(encoding='utf-8'))
    if _client is None:
        _client = OpenAI(api_key=os.environ['DEEPSEEK_API_KEY'], base_url='https://api.deepseek.com')
    extra = {'tools': tools} if tools else {}
    t0 = time.time()
    r = _client.chat.completions.create(model=model, messages=messages, max_tokens=max_tokens, temperature=0, extra_body={'thinking': {'type': 'disabled'}}, **extra)
    m = r.choices[0].message
    out = {'text': (m.content or '').strip(), 'tool_calls': [{'id': t.id, 'name': t.function.name, 'args': t.function.arguments} for t in m.tool_calls or []],
           'in': r.usage.prompt_tokens, 'out': r.usage.completion_tokens, 'sec': round(time.time() - t0, 3), 'model': model, 'fingerprint': r.system_fingerprint}
    p.write_text(json.dumps(out, ensure_ascii=False), encoding='utf-8')
    return out

def ask(kind, prompt, max_tokens, model=GEN_MODEL, run=0):
    return chat(kind, [{'role': 'user', 'content': prompt}], max_tokens, model, run)

def cost(o, model=None):
    pin, pout = PRICE[model or o.get('model', 'claude-sonnet-4-6')]
    return o['in'] * pin + o['out'] * pout

def run_parallel(fn, items, workers=8):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(fn, items))

## 2. Генерация ответов

Промпт генерации: отвечать только по фрагментам, ничего не брать из своих знаний, а если ответа во фрагментах нет, отвечать ровно Не знаю. Если ответ есть только на часть вопроса, отвечать на эту часть и говорить, чего нет. В контекст идут топ-5 чанков вместе с каналом и датой поста, дата нужна для вопросов вроде что было в январе.

Генерирую ответы на все 50 вопросов для всех шести конфигураций поиска. Вот несколько ответов для baseline и best retrieval:

In [7]:
GEN_PROMPT = '''Ты отвечаешь на вопросы по постам из Telegram-каналов про игры. Отвечай только по фрагментам ниже: ничего не добавляй из своих знаний и не додумывай. Если во фрагментах нет ответа на вопрос, ответь ровно: Не знаю. Если ответ есть только на часть вопроса, ответь на эту часть и прямо скажи, чего во фрагментах нет. Пиши по-русски, коротко, без вступлений.

Фрагменты:
{context}

Вопрос: {q}'''

def context_of(idx):
    return '\n\n'.join(f"[{j + 1}] {SRC[chunks[i]['source']]}, {str(chunks[i]['date'])[:10]}\n{chunks[i]['text']}" for j, i in enumerate(idx))

def generate(q, idx):
    return ask('gen', GEN_PROMPT.replace('{context}', context_of(idx)).replace('{q}', q), 600)

def refused(a):
    return a.strip().lower().startswith('не знаю')

jobs = [(name, g) for name in RETRIEVERS for g in gold]
run_parallel(lambda job: generate(job[1]['question'], TOP5[job[0]][job[1]['qid']]), jobs)
ANS = {(name, g['qid']): generate(g['question'], TOP5[name][g['qid']]) for name, g in jobs}
BASE, BEST = 'КТ3: hybrid RRF', 'КТ4: + LLM-реранкер'
print('модель генерации:', GEN_MODEL, '| temperature 0')
for qid in ['q09', 'q16', 'q25']:
    g = next(x for x in gold if x['qid'] == qid)
    print(f"\n{qid} {g['type']} | {g['question']}")
    for name in [BASE, BEST]:
        print(f'   {name}:', ANS[name, qid]['text'][:400])

модель генерации: deepseek-flash | temperature 0

q09 factual | Чё там депутат Михаил Иванов говорил, когда хотел запретить GTA в России, какой аргумент приводил?
   КТ3: hybrid RRF: Депутат Михаил Иванов заявил, что в игре якобы есть деструктивный контент и «угроза традиционным ценностям», из-за чего призвал рассмотреть полную блокировку серии GTA в России.
   КТ4: + LLM-реранкер: Депутат Михаил Иванов заявил, что в игре якобы есть деструктивный контент и «угроза традиционным ценностям», из-за чего призвал рассмотреть полную блокировку серии GTA в России.

q16 composite | Аналитики в январе 2026 года предсказывали выход крупного DLC для The Witcher 3 от CDPR — подтвердился ли их прогноз, и что именно было анонсировано?
   КТ3: hybrid RRF: Прогноз подтвердился: DLC для «Ведьмака 3» анонсировали. По фрагментам, анонсировано дополнение «Песни прошлого» (Songs of the Past) с релизом в 2027 году.
   КТ4: + LLM-реранкер: Прогноз подтвердился: DLC для «Ведьмака 3» анонсировали. По фрагментам

На q09 и q16 топ-5 у обеих конфигураций одинаковый, поэтому и ответы одинаковые. На q25 baseline честно говорит, что цены оригинальной Dying Light во фрагментах нет, а с реранкером модель ответила просто Не знаю.

Теперь негативные вопросы, на которые в корпусе ответа нет:

In [8]:
rows = []
for g in gold:
    rows.append({'qid': g['qid'], 'type': g['type'], 'split': g['split'],
                 **{f'отказ, {name}': refused(ANS[name, g['qid']]['text']) for name in [BASE, BEST]}})
ref = pd.DataFrame(rows)
neg = ref[ref['type'] == 'negative']
print('отказов на негативных:', {name: f"{neg[f'отказ, {name}'].sum()} из {len(neg)}" for name in [BASE, BEST]})
print('отказов на вопросах с ответом:', {name: f"{ref[ref['type'] != 'negative'][f'отказ, {name}'].sum()} из {(ref['type'] != 'negative').sum()}" for name in [BASE, BEST]})
for g in [g for g in gold if g['type'] == 'negative']:
    print(f"\n{g['qid']} | {g['question']}")
    for name in [BASE, BEST]:
        print(f'   {name}:', ANS[name, g['qid']]['text'][:300])

отказов на негативных: {'КТ3: hybrid RRF': '10 из 10', 'КТ4: + LLM-реранкер': '10 из 10'}
отказов на вопросах с ответом: {'КТ3: hybrid RRF': '2 из 40', 'КТ4: + LLM-реранкер': '2 из 40'}

q41 | На каких платформах выйдет Crimson Desert помимо PC?
   КТ3: hybrid RRF: Не знаю.
   КТ4: + LLM-реранкер: Не знаю.

q42 | Какова цена предзаказа «Ведьмак 4»?
   КТ3: hybrid RRF: Не знаю.
   КТ4: + LLM-реранкер: Не знаю.

q43 | кто вообще делает Silent Hill: Townfall?
   КТ3: hybrid RRF: Не знаю.
   КТ4: + LLM-реранкер: Не знаю.

q44 | дата окончания сбора средств Star Citizen
   КТ3: hybrid RRF: Не знаю.
   КТ4: + LLM-реранкер: Не знаю.

q45 | Какие именно сюжетные детали GTA VI нашёл редактор в геймплейном трейлере?
   КТ3: hybrid RRF: Не знаю.
   КТ4: + LLM-реранкер: Не знаю.

q46 | До какого числа продлится Весенняя распродажа Steam 2026?
   КТ3: hybrid RRF: Не знаю.
   КТ4: + LLM-реранкер: Не знаю.

q47 | Слышь, кто-нибудь знает когда уже DirectStorage 2.0 завезут на винду?
   КТ3: hybrid RRF

На всех 10 негативных вопросах обе конфигурации ответили Не знаю, то есть ничего не выдумали. Для сравнения в КТ1 модель без контекста на 4 негативных вопросах из 7 ответила по памяти. Зато на вопросах с ответом модель иногда отказывается зря, по 2 раза из 40: q35 и q37 у baseline, q25 и q37 с реранкером. Там в топ-5 нужной информации правда почти нет.

DeepSeek не даёт закрепить версию точнее, чем имя модели, поэтому смотрю, с какого сервера пришли ответы:

In [9]:
calls = [json.loads(f.read_text(encoding='utf-8')) for f in sorted((CACHE / 'llm').glob('*.json'))]
pd.DataFrame([{'модель': c['model'], 'fingerprint сервера': c['fingerprint']} for c in calls]).value_counts().rename('вызовов').reset_index()

,модель,fingerprint сервера,вызовов
0,deepseek-v4-pro,a307abda487cd1b463329ccb945ce396,2345
1,deepseek-flash,e72d3c7f663d3e1383b00b3884cb4501,580
2,deepseek-flash,bc19ba97cbe016479903222bb1578db6,12


Судья всё время отвечал с одного и того же сервера. У flash 12 вызовов из 592 пришли с другого: 11 шагов агента и одна генерация. Модель та же, но что-то у себя DeepSeek в это время поменял.

## 3. LLM-судья

Три критерия, каждый отдельным запросом и со своей шкалой от 1 до 5, где расписан каждый балл:
- faithfulness: нет ли в ответе того, чего нет в контексте;
- answer relevance: отвечает ли ответ на вопрос;
- context utilization: взял ли ответ всё, что в контексте есть по вопросу.

Судью запускаю только на вопросах с ответом, как и метрики поиска. Негативные проверяю отдельно, по тому, сказала ли модель Не знаю. Промпт переделывал несколько раз, все версии лежат в коде ниже.

In [10]:
JUDGE_HEAD = 'Ты проверяешь ответ RAG-системы, которая отвечает на вопросы по постам из Telegram-каналов про игры. Ниже вопрос, фрагменты контекста, которые получила система, и её ответ.\n\n'
JUDGE_TAIL = '\n\nВопрос: {q}\n\nКонтекст:\n{context}\n\nОтвет системы: {a}\n\nВерни только одно число от 1 до 5.'
JUDGE = {'v1': {
    'faithfulness': JUDGE_HEAD + '''Оцени faithfulness: содержит ли ответ только информацию из контекста, без выдумок.
5 - всё, что сказано в ответе, есть в контексте
4 - почти всё есть в контексте, есть мелкая неточность, которая не меняет смысл
3 - есть одно утверждение, которого нет в контексте
2 - несколько утверждений не из контекста или искажены
1 - ответ в основном выдуман или противоречит контексту''' + JUDGE_TAIL,
    'relevance': JUDGE_HEAD + '''Оцени answer relevance: отвечает ли ответ на поставленный вопрос.
5 - полностью отвечает на вопрос, на все его части
4 - отвечает, но упущена мелкая деталь
3 - отвечает частично
2 - по теме, но на сам вопрос не отвечает
1 - не отвечает на вопрос''' + JUDGE_TAIL,
    'utilization': JUDGE_HEAD + '''Оцени context utilization: использует ли ответ всю информацию из контекста, которая относится к вопросу, или только её часть.
5 - использована вся нужная информация из контекста
4 - упущена мелкая деталь
3 - использована примерно половина нужной информации
2 - использована малая часть
1 - нужная информация в контексте есть, но ответ её не использует''' + JUDGE_TAIL,
}}
JUDGE_TAIL2 = '\n\nВопрос: {q}\n\nКонтекст:\n{context}\n\nОтвет системы: {a}\n\nСначала в 1-3 предложениях разбери ответ по правилам выше, потом последней строкой напиши: Оценка: N'
JUDGE['v2'] = {
    'faithfulness': JUDGE_HEAD + '''Оцени faithfulness: содержит ли ответ только информацию из контекста, без выдумок.
Правила: пересказ, перевод и простые выводы из того, что написано в контексте, считаются информацией из контекста. Факты из общих знаний, которых нет в контексте, считаются выдумкой, даже если они верны. Ответ «Не знаю» и фразы вида «в контексте нет X» не выдумка.
5 - всё, что сказано в ответе, есть в контексте
4 - всё по сути из контекста, но есть мелкая неточность, которая не меняет смысл (перепутан порядок, округлено число)
3 - есть одно утверждение, которого нет в контексте или которое искажено
2 - несколько утверждений не из контекста или искажены
1 - ответ в основном выдуман или противоречит контексту''' + JUDGE_TAIL2,
    'relevance': JUDGE_HEAD + '''Оцени answer relevance: отвечает ли ответ на поставленный вопрос.
Правила: оценивай только то, отвечает ли ответ на вопрос, а не то, верен ли он. Если в вопросе несколько частей, смотри, на сколько из них есть конкретный ответ. Часть, про которую сказано, что информации нет, считается неотвеченной.
5 - есть конкретный ответ на все части вопроса
4 - есть ответ на все части, но одна из них раскрыта неполно или расплывчато
3 - есть ответ только на часть вопроса, про остальное сказано, что информации нет
2 - есть только косвенная информация по теме, на сам вопрос ответа нет
1 - ответа нет совсем: «Не знаю» или только объяснение, почему ответить нельзя''' + JUDGE_TAIL2,
    'utilization': JUDGE_HEAD + '''Оцени context utilization: использует ли ответ всю информацию из контекста, которая относится к вопросу, или только её часть.
Правила: сначала определи, какие фрагменты контекста содержат информацию по вопросу. Оценивай только относительно них и не снижай оценку за то, чего в контексте нет. Если в контексте нет ничего по вопросу и ответ это говорит, это 5.
5 - ответ использует всё, что в контексте относится к вопросу
4 - упущена одна мелкая деталь из нужных фрагментов
3 - упущен один из нужных фрагментов или важная деталь
2 - использована малая часть нужной информации
1 - нужная информация в контексте есть, но ответ её не использует или говорит «Не знаю»''' + JUDGE_TAIL2,
}
JUDGE_TAIL3 = '\n\nВопрос: {q}\n\nКонтекст:\n{context}\n\nОтвет системы: {a}\n\nСначала сделай разбор, как сказано выше, коротко, по одной строке на пункт. Последней строкой напиши: Оценка: N'
JUDGE['v3'] = {
    'faithfulness': JUDGE_HEAD + '''Оцени faithfulness: содержит ли ответ только информацию из контекста, без выдумок.
Разбор: выпиши по очереди утверждения ответа и у каждого отметь: есть в контексте / неточно / нет в контексте. Пересказ, перевод и простые выводы из контекста считаются «есть». Факты из общих знаний, которых нет в контексте, считаются «нет», даже если они верны. «Не знаю» и фразы вида «в контексте нет X» не утверждения, их не считай.
5 - все утверждения есть в контексте, или утверждений нет
4 - одно утверждение неточно (перепутан порядок, округлено число), остальные есть
3 - одного утверждения нет в контексте
2 - двух и больше утверждений нет в контексте
1 - большей части утверждений нет в контексте или ответ противоречит контексту''' + JUDGE_TAIL3,
    'relevance': JUDGE_HEAD + '''Оцени answer relevance: отвечает ли ответ на поставленный вопрос. Верен ли ответ и откуда он взят, здесь не важно.
Разбор: выпиши, о чём по отдельности спрашивает вопрос (одна или несколько частей), и у каждой части отметь: ответ есть / частично / нет. «Частично» - ответ общий или неполный, например спрашивают про три игры, а названа одна. Если про часть сказано, что информации нет, это «нет».
5 - у всех частей ответ есть
4 - у всех частей ответ есть или частично, и хотя бы одна частично
3 - хотя бы у одной части ответ есть или частично, и хотя бы у одной нет
2 - нет ни одной части с ответом, но есть информация по теме вопроса
1 - ответа нет совсем: «Не знаю» или только объяснение, почему ответить нельзя''' + JUDGE_TAIL3,
    'utilization': JUDGE_HEAD + '''Оцени context utilization: использует ли ответ всю информацию из контекста, которая относится к вопросу, или только её часть.
Разбор: выпиши номера фрагментов, где есть информация по вопросу, и у каждого отметь: использован полностью / частично / не использован. Не снижай оценку за то, чего в контексте нет.
5 - все нужные фрагменты использованы полностью, или нужных фрагментов нет и ответ говорит, что не знает
4 - все нужные фрагменты использованы, но хотя бы один частично
3 - один нужный фрагмент не использован
2 - не использовано больше половины нужных фрагментов
1 - нужные фрагменты есть, но ни один не использован''' + JUDGE_TAIL3,
}
JUDGE['v4'] = {
    'faithfulness': JUDGE['v2']['faithfulness'],
    'relevance': JUDGE_HEAD + '''Оцени answer relevance: отвечает ли ответ на поставленный вопрос.
Правила: оценивай только то, отвечает ли ответ на вопрос. Верен ли он, откуда взят и всё ли из контекста в него вошло, здесь не важно, это проверяют другие критерии. Если в вопросе несколько частей, смотри, на сколько из них есть конкретный ответ. Смотри на содержание, а не на оговорки: если ответ пишет, что информации нет, но дальше всё-таки отвечает по существу, считай по существу. Если вопрос просит перечислить (какие игры, какие скидки), перечень считается ответом. Если вопрос просит мнение или прогноз, прогноз или факты по этой теме считаются ответом.
5 - есть конкретный ответ на все части вопроса
4 - есть ответ на все части, но одна из них раскрыта расплывчато
3 - есть ответ только на часть вопроса, про остальное сказано, что информации нет
2 - есть только косвенная информация по теме, на сам вопрос ответа нет
1 - ответа нет совсем: «Не знаю» или только объяснение, почему ответить нельзя''' + JUDGE_TAIL2,
    'utilization': JUDGE_HEAD + '''Оцени context utilization: использует ли ответ всю информацию из контекста, которая нужна для ответа на вопрос, или только её часть.
Правила: нужная информация - это только факты, которые прямо отвечают на вопрос или на его часть. Подробности, которые просто касаются той же темы (другие события, последствия, оформление), не нужны, и если их нет в ответе, оценку не снижай. Если ответ взял из фрагмента всё нужное, что там есть, фрагмент использован полностью, даже если из-за этого ответ неполный. Если нужной информации в контексте нет и ответ это говорит, это 5.
5 - в ответе есть вся нужная информация из контекста
4 - упущена одна мелкая нужная деталь
3 - упущен один нужный факт
2 - упущено больше половины нужных фактов
1 - нужная информация в контексте есть, но ответ её не использует или говорит «Не знаю»''' + JUDGE_TAIL2,
}
CRITERIA = ['faithfulness', 'relevance', 'utilization']
JUDGE_VERSION = 'v1'

def parse_score(text):
    m = re.findall(r'Оценка:\s*([1-5])', text) or re.findall(r'\b([1-5])\b', text)
    return int(m[-1]) if m else None

def judge(criterion, q, idx, a, version=None, run=0):
    prompt = JUDGE[version or JUDGE_VERSION][criterion].replace('{q}', q).replace('{context}', context_of(idx)).replace('{a}', a)
    o = ask(f'judge_{criterion}', prompt, 700, JUDGE_MODEL, run)
    if parse_score(o['text']) is None:
        retry = ask(f'judge_{criterion}', prompt + '\n\nОбязательно закончи ответ строкой: Оценка: N', 700, JUDGE_MODEL, run)
        o = {**retry, 'in': o['in'] + retry['in'], 'out': o['out'] + retry['out'], 'sec': o['sec'] + retry['sec']}
    return {**o, 'score': parse_score(o['text'])}

### Проверка стабильности судьи

Беру 30 случайных пар вопрос и ответ из val, обе конфигурации вперемешку, и прогоняю каждую версию судьи по два раза при temperature 0. Agreement это доля пар, где оценка во второй раз совпала с первой. Нужно 85% и выше по каждому критерию. Ниже agreement по версиям и все расхождения.

In [11]:
pairs, seen = [], set()
for g in val_q:
    for name in [BASE, BEST]:
        key = (g['qid'], tuple(TOP5[name][g['qid']]))
        if g['type'] != 'negative' and key not in seen:
            seen.add(key)
            pairs.append({'qid': g['qid'], 'q': g['question'], 'idx': TOP5[name][g['qid']], 'a': ANS[name, g['qid']]['text']})
pairs = random.Random(42).sample(pairs, 30)

def validate(version):
    jobs = [(p, c, run) for p in pairs for c in CRITERIA for run in (1, 2)]
    run_parallel(lambda j: judge(j[1], j[0]['q'], j[0]['idx'], j[0]['a'], version, j[2]), jobs)
    rows = []
    for p in pairs:
        for c in CRITERIA:
            a, b = (judge(c, p['q'], p['idx'], p['a'], version, run) for run in (1, 2))
            rows.append({'qid': p['qid'], 'criterion': c, 'run1': a['score'], 'run2': b['score'], 'text1': a['text'], 'text2': b['text'], 'pair': p})
    return pd.DataFrame(rows)

VAL_RUNS = {version: validate(version) for version in JUDGE}
pd.DataFrame({version: df.groupby('criterion').apply(lambda d: f"{100 * (d['run1'] == d['run2']).mean():.0f}%") for version, df in VAL_RUNS.items()}).T

criterion,faithfulness,relevance,utilization
v1,100%,93%,90%
v2,100%,90%,93%
v3,100%,90%,83%
v4,100%,87%,80%


In [12]:
def show_diffs(version, n=None):
    df = VAL_RUNS[version]
    for _, r in df[df['run1'] != df['run2']].head(n).iterrows():
        reason = re.sub(r'\s+', ' ', r['text1'].split('Оценка')[0])[:300]
        print(f"{version} | {r['qid']} | {r['criterion']} | {r['run1']} и {r['run2']} | ответ: {r['pair']['a'][:150]}")
        if re.search('[а-яё]', reason.lower()):
            print('   разбор первого прогона:', reason)

for version in JUDGE:
    show_diffs(version)
    print()

v1 | q18 | relevance | 5 и 4 | ответ: К 2028 году Sony прекратит выпускать игры для PlayStation на дисках: все новые релизы будут выходить только в PlayStation Store или в виде цифровых кл
v1 | q33 | utilization | 4 и 5 | ответ: Во фрагментах указаны скидки: -80%, -70%, -60%, -95%, -70%. Наибольшая скидка — -95% (55 рублей, пост [4]). Какие именно игры получили эти скидки, во 
v1 | q01 | utilization | 5 и 4 | ответ: Во фрагментах указана только одна игра из подборки PS Plus за январь 2026 — Need for Speed Unbound (PS5). Остальные две игры не названы.
v1 | q16 | utilization | 4 и 5 | ответ: Прогноз подтвердился: DLC для «Ведьмака 3» анонсировали. По фрагментам, анонсировано дополнение «Песни прошлого» (Songs of the Past) с релизом в 2027 
v1 | q36 | relevance | 1 и 2 | ответ: Во фрагментах нет названий конкретных игр из подборок со скидками — упоминаются только сами подборки ([1], [2], [4]) без перечисления игр. Поэтому отв

v2 | q17 | relevance | 4 и 3 | ответ: Во фрагментах нет сведен

Как менялся промпт:
- **v1**: просто шкала и ответ одним числом. Порог по agreement уже пройден, но все 5 расхождений из-за дыр в шкале. q36 по relevance то 1, то 2: модель не ответила, но объяснила почему, а в шкале не сказано, сколько ставить за такой отказ. q18 по relevance то 5, то 4: на вторую часть вопроса ответ с оговоркой, что во фрагментах про это мало, и непонятно, считать ли эту часть отвеченной. q01 и q33 по utilization: нужной информации в контексте просто нет (в q01 названа только одна игра из трёх), а в шкале не сказано, снижать ли за это. q16 по utilization: непонятно, какие фрагменты считать нужными, если про одно и то же пишут три поста.
- **v2**: добавил правила под эти случаи. Не знаю и объяснение без ответа это 1 по relevance. Часть вопроса, про которую сказано, что информации нет, считается неотвеченной. За то, чего нет в контексте, utilization не снижать. И попросил перед оценкой коротко разобрать ответ. Эти 5 расхождений ушли. Новые появились на других парах: где ответ закрывает вопрос не полностью и где непонятно, какой фрагмент считать нужным (q40, q18).
- **v3**: попробовал заставить судью раскладывать ответ по пунктам: по relevance выписать части вопроса и отметить у каждой, есть ли ответ, по utilization отметить каждый фрагмент. Utilization упал ниже порога. Видно по разбору q18: судья отмечает фрагмент как использованный частично из-за деталей, которые к вопросу не относятся (пометка на коробке PS5, закрытие PS Store на PS Vita), и в одном прогоне таких частично больше, чем в другом.
- **v4**: вернулся к v2 и добавил ещё правил: что считать нужной информацией, не смотреть на оговорки в ответе, как оценивать вопросы про мнение. Стало хуже: правила начали спорить друг с другом. На q25 и q37 ответ Не знаю, а в контексте есть кусок информации, и судья то ставит 1 по правилу про неиспользованный контекст, то 5 по правилу, что нужной информации нет.

Беру v2. По agreement v1 и v2 почти одинаковые, разница в одну пару, но v2 закрыла все случаи, где шкала v1 была непонятной, и у неё виден разбор, за что поставлена оценка. Добавление правил дальше только ухудшало стабильность. Для utilization потом пришлось сделать ещё версии, после ручной проверки, это ниже. Faithfulness во всех версиях совпадал на 100%, но тут все 30 ответов получили 5, так что стабильность тут ничего не доказывает.

In [13]:
used_keys = {(p['qid'], tuple(p['idx'])) for p in pairs}
pool, seen = [], set(used_keys)
for g in val_q:
    for name in RETRIEVERS:
        key = (g['qid'], tuple(TOP5[name][g['qid']]))
        if g['type'] != 'negative' and key not in seen:
            seen.add(key)
            pool.append({'qid': g['qid'], 'q': g['question'], 'idx': TOP5[name][g['qid']], 'a': ANS[name, g['qid']]['text']})
holdout = random.Random(1).sample(pool, 30)
jobs = [(p, c, run) for p in holdout for c in CRITERIA for run in (1, 2)]
run_parallel(lambda j: judge(j[1], j[0]['q'], j[0]['idx'], j[0]['a'], 'v2', j[2]), jobs)
print('v2 на 30 других парах из val, agreement:', {c: f"{100 * np.mean([judge(c, p['q'], p['idx'], p['a'], 'v2', 1)['score'] == judge(c, p['q'], p['idx'], p['a'], 'v2', 2)['score'] for p in holdout]):.0f}%" for c in CRITERIA})

v2 на 30 других парах из val, agreement: {'faithfulness': '100%', 'relevance': '93%', 'utilization': '87%'}


v2 я правил под расхождения на тех же 30 парах, поэтому проверил её ещё на 30 других парах из val, это ответы остальных конфигураций. Там agreement тоже выше 85% по всем трём критериям.

### Оценки ответов

Прогоняю судью v2 по ответам всех шести конфигураций. Сначала две главные, потом по типам вопросов на val:

In [14]:
JUDGE_VERSION = 'v2'
answerable = [g for g in gold if g['type'] != 'negative']
jobs = [(name, g, c) for name in RETRIEVERS for g in answerable for c in CRITERIA]
run_parallel(lambda j: judge(j[2], j[1]['question'], TOP5[j[0]][j[1]['qid']], ANS[j[0], j[1]['qid']]['text']), jobs)
SCORE = {(name, g['qid'], c): judge(c, g['question'], TOP5[name][g['qid']], ANS[name, g['qid']]['text'])['score'] for name, g, c in jobs}

def gen_metrics(name, qs):
    pos = [g for g in qs if g['type'] != 'negative']
    neg = [g for g in qs if g['type'] == 'negative']
    out = {c: round(float(np.mean([SCORE[name, g['qid'], c] for g in pos])), 2) for c in CRITERIA}
    out['отказы на негативных'] = f"{sum(refused(ANS[name, g['qid']]['text']) for g in neg)} из {len(neg)}"
    out['ложные отказы'] = f"{sum(refused(ANS[name, g['qid']]['text']) for g in pos)} из {len(pos)}"
    return out

pd.DataFrame([{'контекст': name, 'split': split, **gen_metrics(name, qs)} for name in [BASE, BEST] for split, qs in [('val', val_q), ('test', test_q)]])

,контекст,split,faithfulness,relevance,utilization,отказы на негативных,ложные отказы
0,КТ3: hybrid RRF,val,4.93,3.71,4.61,7 из 7,1 из 28
1,КТ3: hybrid RRF,test,5.00,4.17,4.75,3 из 3,1 из 12
2,КТ4: + LLM-реранкер,val,5.00,3.86,4.43,7 из 7,2 из 28
3,КТ4: + LLM-реранкер,test,5.00,4.42,4.92,3 из 3,0 из 12


In [15]:
rows = []
for name in [BASE, BEST]:
    for t in ['factual', 'composite', 'abstract']:
        qs = [g for g in val_q if g['type'] == t]
        rows.append({'контекст': name, 'тип': t, 'вопросов': len(qs), **{c: round(np.mean([SCORE[name, g['qid'], c] for g in qs]), 2) for c in CRITERIA},
                     'R@5': round(RET[name][0].set_index('qid').loc[[g['qid'] for g in qs], 'R@5'].mean(), 2)})
pd.DataFrame(rows)

,контекст,тип,вопросов,faithfulness,relevance,utilization,R@5
0,КТ3: hybrid RRF,factual,11,4.82,4.82,4.55,1.00
1,КТ3: hybrid RRF,composite,10,5.00,3.10,4.60,0.97
2,КТ3: hybrid RRF,abstract,7,5.00,2.86,4.71,0.02
3,КТ4: + LLM-реранкер,factual,11,5.00,4.82,4.55,1.00
4,КТ4: + LLM-реранкер,composite,10,5.00,3.60,4.50,1.00
5,КТ4: + LLM-реранкер,abstract,7,5.00,2.71,4.14,0.05


Ответы почти никогда не выдумывают: faithfulness около 5 у обеих конфигураций. Проблема в relevance: на val 3.71 у baseline и 3.86 с реранкером, то есть ответ часто неполный. По типам видно где. На фактических вопросах всё хорошо. На составных relevance около 3, хотя поиск находит почти все нужные посты, значит теряется уже после поиска. На абстрактных поиск почти ничего не находит, и ответ тоже слабый. Реранкер немного помогает составным, на остальном разницы нет.

## 4. Агентский RAG

Выбрал агентский RAG. У агента три инструмента: векторный поиск (FRIDA и Chroma из КТ3-КТ4), BM25 из КТ3 и уточняющий вопрос пользователю. Сам агент на deepseek-flash, как и обычная генерация, чтобы сравнение было честным.

### Набор сложных вопросов

Нужны вопросы, где надо свести информацию из разных постов. Делаю так. Считаю эмбеддинг каждого поста как среднее его чанков и ищу группы похожих постов про одно и то же, но с разными датами. По каждой группе судейская модель придумывает вопрос, которому нужны минимум два поста, и ответ. Потом отдельный запрос проверяет, что ответ верный, что одного поста правда не хватает и что вопрос понятен без постов.

Ещё нужны неоднозначные вопросы для уточняющего вопроса. Их делаю по одному посту: вопрос без главной детали, вроде что там сломалось после обновления, и что пользователь ответит, если его переспросить. Ask_user в тесте возвращает этот заготовленный ответ, то есть пользователя я имитирую.

In [16]:
post_ids = [d['doc_id'] for d in docs_clean]
post_type = [content_type(d['text']) for d in docs_clean]
post_date = pd.to_datetime([d['date'] for d in docs_clean])
V = np.stack([E[chunk_doc == p].mean(0) for p in post_ids])
V /= np.linalg.norm(V, axis=1, keepdims=True)
S = V @ V.T

def post_block(j, n):
    d = docs_clean[j]
    return f"[{n}] {str(d['date'])[:10]}, {SRC[d['source']]}\n{d['text']}"

rng = random.Random(42)
seeds = [j for j in range(len(docs_clean)) if post_type[j] == 'news' and len(docs_clean[j]['text']) >= 150]
rng.shuffle(seeds)
used, groups = set(), []
for j in seeds:
    if j in used:
        continue
    partners = [k for k in np.argsort(-S[j]) if k != j and k not in used and post_type[k] in ('news', 'giveaway', 'deal')
                and 0.6 <= S[j, k] <= 0.95 and abs((post_date[k] - post_date[j]).days) >= 2][:2]
    if partners:
        group = sorted([j] + partners, key=lambda k: post_date[k])
        used.update(group)
        groups.append(group)
    if len(groups) == 60:
        break
amb_seeds = [j for j in seeds if j not in used][:12]
print('групп постов:', len(groups), '| из них по 3 поста:', sum(len(g) == 3 for g in groups), '| постов для неоднозначных вопросов:', len(amb_seeds))

групп постов: 60 | из них по 3 поста: 34 | постов для неоднозначных вопросов: 12


In [17]:
MH_PROMPT = '''Ниже несколько постов из Telegram-каналов про игры на близкую тему, с датами. Придумай один сложный вопрос, на который можно ответить, только если свести вместе информацию минимум из двух постов: сравнить что-то, проследить, как менялась ситуация со временем, связать причину и следствие или собрать сводку. Вопрос должен быть понятен без постов: называй игры, компании и события, но не пиши «в посте» или «по данным канала». Не подсказывай ответ в самом вопросе. Пиши так, как спросил бы обычный пользователь.

Если по этим постам нельзя сделать хороший вопрос, которому нужны несколько постов, верни {"skip": true}. Иначе верни только JSON:
{"question": "...", "answer": "короткий правильный ответ по постам", "posts": [номера нужных постов], "kind": "сравнение" или "развитие событий" или "причина и следствие" или "сводка"}

Посты:
{posts}'''

AMB_PROMPT = '''Ниже пост из Telegram-канала про игры. Придумай вопрос пользователя, который без уточнения неоднозначен: в нём не хватает главной детали (не названа игра, непонятно, о какой распродаже, раздаче или компании речь), так что под него подходят многие разные посты. Ещё придумай, что пользователь ответит, если его переспросить: короткое уточнение, после которого ясно, что речь о событии из этого поста.

Верни только JSON: {"question": "...", "clarification": "...", "answer": "короткий ответ по посту"}

Пост:
{posts}'''

VERIFY_PROMPT = '''Проверь вопрос для теста поисковой системы по постам про игры. Ниже вопрос, эталонный ответ и посты, по которым он составлен.
Проверь три вещи:
correct - эталонный ответ верный и полностью следует из постов;
multi - для ответа нужны минимум два разных поста, одного поста не хватает;
clear - вопрос понятен без постов и не подсказывает ответ.
Верни только JSON: {"correct": true или false, "multi": true или false, "clear": true или false, "comment": "одной фразой, что не так"}

Вопрос: {q}
Эталонный ответ: {a}

Посты:
{posts}'''

def parse_json(text):
    return json.loads(re.search(r'\{.*\}', text, re.S).group(0))

def mh_prompt(group):
    return MH_PROMPT.replace('{posts}', '\n\n'.join(post_block(j, n + 1) for n, j in enumerate(group)))

def amb_prompt(j):
    return AMB_PROMPT.replace('{posts}', post_block(j, 1))

run_parallel(lambda g: ask('agent_set', mh_prompt(g), 600, JUDGE_MODEL), groups)
run_parallel(lambda j: ask('agent_set', amb_prompt(j), 400, JUDGE_MODEL), amb_seeds)
raw = []
for group in groups:
    x = parse_json(ask('agent_set', mh_prompt(group), 600, JUDGE_MODEL)['text'])
    if not x.get('skip'):
        raw.append({**x, 'doc_ids': [post_ids[group[n - 1]] for n in x['posts']], 'group': group})
print('групп:', len(groups), '| вопросов придумано:', len(raw), '| пропущено:', len(groups) - len(raw))

def verify_prompt(x):
    return VERIFY_PROMPT.replace('{q}', x['question']).replace('{a}', x['answer']).replace('{posts}', '\n\n'.join(post_block(j, n + 1) for n, j in enumerate(x['group'])))

run_parallel(lambda x: ask('agent_set', verify_prompt(x), 300, JUDGE_MODEL), raw)
for x in raw:
    x['check'] = parse_json(ask('agent_set', verify_prompt(x), 300, JUDGE_MODEL)['text'])
ok = [x for x in raw if x['check']['correct'] and x['check']['multi'] and x['check']['clear']]
print('прошли проверку:', len(ok), 'из', len(raw))
for x in [x for x in raw if x not in ok][:6]:
    print('   отброшен:', x['question'][:90], '|', x['check']['comment'][:120])

групп: 60 | вопросов придумано: 60 | пропущено: 0
прошли проверку: 51 из 60
   отброшен: Какое крупное обновление для GTA Online вышло 14 июля 2026 года и что в нём предстояло сде | Вся информация для ответа содержится в одном посте [1], поэтому условие multi не выполняется.
   отброшен: Какие события, произошедшие с интервалом менее чем в два месяца в 2026 году, могли привест | Вопрос неясен и содержит ложную предпосылку: посты не описывают события 2026 года с интервалом менее двух месяцев, а пос
   отброшен: Во сколько раз коллекционное издание GTA VI за $400 дороже стандартного физического издани | Курс на момент анонса коллекционки в постах не указан, поэтому эталонный ответ 'примерно в 3,4 раза' не следует из предо
   отброшен: Какая крупная однопользовательская игра от внутренней студии Sony, по слухам, не выйдет на | Посты утверждают, что Marvel's Wolverine как раз НЕ выйдет на ПК, а не наоборот, и для ответа достаточно одного поста.
   отброшен: Какая игра, вышедшая в 2016 году

In [18]:
VERIFY_AMB_PROMPT = '''Проверь вопрос для теста поисковой системы по постам про игры. Пользователь задал неоднозначный вопрос, а когда его переспросили, уточнил. Ниже вопрос, уточнение, эталонный ответ и пост, по которому всё составлено.
Проверь три вещи:
ambiguous - без уточнения вопрос действительно неоднозначен, под него подходят разные события или игры;
resolved - с уточнением понятно, что речь именно о событии из поста;
correct - эталонный ответ верный и следует из поста.
Верни только JSON: {"ambiguous": true или false, "resolved": true или false, "correct": true или false, "comment": "одной фразой, что не так"}

Вопрос: {q}
Уточнение: {c}
Эталонный ответ: {a}

Пост:
{posts}'''

def verify_amb_prompt(x):
    return VERIFY_AMB_PROMPT.replace('{q}', x['question']).replace('{c}', x['clarification']).replace('{a}', x['answer']).replace('{posts}', post_block(x['seed'], 1))

amb = [{**parse_json(ask('agent_set', amb_prompt(j), 400, JUDGE_MODEL)['text']), 'seed': j, 'doc_ids': [post_ids[j]]} for j in amb_seeds]
run_parallel(lambda x: ask('agent_set', verify_amb_prompt(x), 300, JUDGE_MODEL), amb)
for x in amb:
    x['check'] = parse_json(ask('agent_set', verify_amb_prompt(x), 300, JUDGE_MODEL)['text'])
amb_ok = [x for x in amb if x['check']['ambiguous'] and x['check']['resolved'] and x['check']['correct']]
print('неоднозначных прошли проверку:', len(amb_ok), 'из', len(amb))

MANUAL_DROP = {
    'Какая технология, упомянутая в конце августа': 'склеены два несвязанных факта, ответ только на первую часть',
    'А что за игра сейчас бесплатно в Steam': 'пост не про раздачу, вопрос не имеет смысла',
    'Что там за скандал с читами у Blizzard': 'компания названа, вопрос не неоднозначный',
}
keep = lambda x: not any(x['question'].startswith(k) for k in MANUAL_DROP)
rng = random.Random(42)
mh = rng.sample([x for x in ok if keep(x)], 40)
agent_set = [{'question': x['question'], 'answer': x['answer'], 'doc_ids': x['doc_ids'], 'kind': x['kind'], 'clarification': None} for x in mh]
agent_set += [{'question': x['question'], 'answer': x['answer'], 'doc_ids': x['doc_ids'], 'kind': 'неоднозначный', 'clarification': x['clarification']} for x in amb_ok if keep(x)]
order = list(range(len(agent_set)))
rng.shuffle(order)
seen_kind = {}
for n, i in enumerate(order):
    k = agent_set[i]['kind']
    seen_kind[k] = seen_kind.get(k, 0) + 1
    agent_set[i]['aid'] = f'a{n + 1:02d}'
    agent_set[i]['split'] = 'test' if seen_kind[k] % 3 == 0 else 'val'
agent_set.sort(key=lambda x: x['aid'])
json.dump(agent_set, open(DATA / 'agent_set.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('вопросов:', len(agent_set), '| val:', sum(x['split'] == 'val' for x in agent_set), '| test:', sum(x['split'] == 'test' for x in agent_set))
pd.crosstab(pd.Series([x['kind'] for x in agent_set], name='тип'), pd.Series([x['split'] for x in agent_set], name='split'))

неоднозначных прошли проверку: 12 из 12
вопросов: 50 | val: 35 | test: 15


split,test,val
тип,,
неоднозначный,3,7
причина и следствие,0,2
развитие событий,5,10
сводка,6,12
сравнение,1,4


Из 60 многошаговых вопросов проверку прошёл 51, остальные отброшены, в основном потому что хватало одного поста. Неоднозначные прошли все 12. Потом я прочитал все вопросы и выкинул ещё три (они в коде с причиной). Из оставшихся многошаговых случайно взял 40, чтобы всего было 50 вопросов: 40 многошаговых и 10 неоднозначных, 35 в val и 15 в test.

### Агент

Агент получает вопрос и сам решает, какие инструменты звать. В системном промпте сказано: неоднозначный вопрос сначала уточнить, сложный разбить на части, названия и цифры искать через BM25, смысл через векторный поиск, перед каждым вызовом одной фразой написать зачем, отвечать только по найденному, иначе Не знаю. Число поисков ограничено. Прогоняю агента на новом наборе и на золотом. Время это реальное время ответов DeepSeek плюс время поиска. Ответ пользователя на уточнение в задержку не входит, его я имитирую.

In [19]:
AGENT_SYSTEM = '''Ты отвечаешь на вопросы по постам из двух Telegram-каналов про игры: раздачи, скидки, новости индустрии за 2025-2026 годы. У тебя есть два поиска по постам и уточняющий вопрос пользователю.

Как работать:
- Если вопрос неоднозначный и непонятно, о какой игре, компании или событии речь, сначала задай уточняющий вопрос через ask_user.
- Сложный вопрос разбей на части и ищи каждую отдельно. Названия, цифры и даты лучше искать через search_bm25, смысл и перефразировки через search_vector. Если поиск не нашёл нужного, переформулируй запрос или попробуй другой поиск.
- Перед каждым вызовом инструмента одной фразой напиши, что ищешь и почему этим инструментом.
- Делай не больше 6 поисков.
- Отвечай только по найденным фрагментам, ничего не добавляй из своих знаний. Если ответа во фрагментах нет, ответь ровно: Не знаю. Если нашёл ответ только на часть вопроса, ответь на неё и скажи, чего не нашёл.
- Итоговый ответ пиши по-русски, коротко, без вступлений.'''

def tool(name, description, arg, arg_description):
    return {'type': 'function', 'function': {'name': name, 'description': description, 'parameters': {
        'type': 'object', 'properties': {arg: {'type': 'string', 'description': arg_description}}, 'required': [arg]}}}

TOOLS = [
    tool('search_vector', 'Смысловой поиск по постам: FRIDA и Chroma из КТ3-КТ4. Находит посты по смыслу, даже если слова другие. Возвращает 5 фрагментов.', 'query', 'поисковый запрос'),
    tool('search_bm25', 'Поиск по словам, BM25 из КТ3. Хорошо находит точные названия игр и компаний, цифры и даты. Возвращает 5 фрагментов.', 'query', 'поисковый запрос'),
    tool('ask_user', 'Уточняющий вопрос пользователю, если непонятно, о какой игре, компании или событии он спрашивает. Возвращает ответ пользователя.', 'question', 'вопрос пользователю'),
]
MAX_SEARCHES, MAX_CALLS = 6, 10
TOOL_TIME = TIMES.setdefault('agent_tools', {})

def run_tool(name, args, clarification):
    if name == 'ask_user':
        return (clarification or 'Пользователь не стал уточнять. Попробуй найти ответ сам.'), [], 0.0
    q = args.get('query', '')
    with LOCK:
        t0 = time.perf_counter()
        idx = list(map(int, (bm25_list(q) if name == 'search_bm25' else dense_list(q))[:5]))
        if (name, q) not in TOOL_TIME:
            TOOL_TIME[name, q] = 1000 * (time.perf_counter() - t0)
            save_pkl(TIMES, 'timings.pkl')
    text = '\\n\\n'.join(f"[фрагмент {i}] {SRC[chunks[i]['source']]}, {str(chunks[i]['date'])[:10]}\\n{chunks[i]['text']}" for i in idx)
    return text, idx, TOOL_TIME[name, q]

def run_agent(question, clarification=None):
    messages = [{'role': 'system', 'content': AGENT_SYSTEM}, {'role': 'user', 'content': question}]
    calls, trace, found, searches, tool_ms = [], [], [], 0, 0.0
    answer = None
    for _ in range(MAX_CALLS):
        o = chat('agent', messages, 800, GEN_MODEL, tools=TOOLS)
        calls.append(o)
        if not o['tool_calls']:
            answer = o['text']
            break
        messages.append({'role': 'assistant', 'content': o['text'], 'tool_calls': [
            {'id': t['id'], 'type': 'function', 'function': {'name': t['name'], 'arguments': t['args']}} for t in o['tool_calls']]})
        for t in o['tool_calls']:
            args = json.loads(t['args'] or '{}')
            if t['name'] != 'ask_user' and searches >= MAX_SEARCHES:
                result, idx, ms = 'Лимит поисков исчерпан, отвечай по тому, что уже нашёл.', [], 0.0
            else:
                result, idx, ms = run_tool(t['name'], args, clarification)
                searches += t['name'] != 'ask_user'
            tool_ms += ms
            found += [i for i in idx if i not in found]
            trace.append({'thought': o['text'], 'tool': t['name'], 'arg': next(iter(args.values()), ''), 'idx': idx, 'result': result if t['name'] == 'ask_user' else ''})
            messages.append({'role': 'tool', 'tool_call_id': t['id'], 'content': result})
    if answer is None:
        o = chat('agent', messages + [{'role': 'user', 'content': 'Поиски закончились. Дай итоговый ответ по найденным фрагментам.'}], 800, GEN_MODEL)
        calls.append(o)
        answer = o['text']
    return {'answer': answer, 'trace': trace, 'context': found, 'llm_calls': len(calls), 'tool_calls': len(trace),
            'sec': sum(c['sec'] for c in calls) + tool_ms / 1000, 'cost': sum(cost(c) for c in calls)}

agent_set = json.load(open(DATA / 'agent_set.json', encoding='utf-8'))
AGENT = dict(zip([x['aid'] for x in agent_set], run_parallel(lambda x: run_agent(x['question'], x['clarification']), agent_set)))
AGENT_GOLD = dict(zip([g['qid'] for g in gold], run_parallel(lambda g: run_agent(g['question']), gold)))
print('агент: вопросов', len(AGENT), '+ золотой набор', len(AGENT_GOLD))

агент: вопросов 50 + золотой набор 50


In [20]:
def agent_row(runs, items, label):
    tools = pd.Series([t['tool'] for r in runs for t in r['trace']]).value_counts()
    sec = np.array([r['sec'] for r in runs])
    return {'набор': label, 'вопросов': len(runs), 'вызовов инструментов, среднее': round(np.mean([r['tool_calls'] for r in runs]), 2),
            'вызовов LLM, среднее': round(np.mean([r['llm_calls'] for r in runs]), 2),
            **{t: int(tools.get(t, 0)) for t in ['search_bm25', 'search_vector', 'ask_user']},
            'медиана, с': round(np.median(sec), 2), 'p95, с': round(np.percentile(sec, 95), 2), 'стоимость $ на запрос': round(np.mean([r['cost'] for r in runs]), 4)}

amb_ids = [x['aid'] for x in agent_set if x['clarification']]
other_ids = [x['aid'] for x in agent_set if not x['clarification']]
asked = lambda r: any(t['tool'] == 'ask_user' for t in r['trace'])
print('ask_user на неоднозначных:', sum(asked(AGENT[a]) for a in amb_ids), 'из', len(amb_ids), '| на остальных:', sum(asked(AGENT[a]) for a in other_ids), 'из', len(other_ids),
      '| на золотом наборе:', sum(asked(r) for r in AGENT_GOLD.values()), 'из', len(AGENT_GOLD))
pd.DataFrame([agent_row([AGENT[a] for a in other_ids], None, 'многошаговые вопросы'), agent_row([AGENT[a] for a in amb_ids], None, 'неоднозначные'),
              agent_row(list(AGENT_GOLD.values()), None, 'золотой набор')])

ask_user на неоднозначных: 8 из 10 | на остальных: 4 из 40 | на золотом наборе: 3 из 50


,набор,вопросов,"вызовов инструментов, среднее","вызовов LLM, среднее",search_bm25,search_vector,ask_user,"медиана, с","p95, с",стоимость $ на запрос
0,многошаговые вопросы,40,3.68,2.88,79,64,4,4.33,7.00,0.0020
1,неоднозначные,10,3.00,2.90,11,11,8,4.06,5.07,0.0015
2,золотой набор,50,3.04,2.54,77,72,3,3.04,6.96,0.0017


На многошаговом вопросе агент в среднем зовёт инструменты 3-4 раза за 3 обращения к LLM. Часто он сразу просит два поиска разом, BM25 и векторный, потом смотрит, чего не хватает, и ищет ещё. Ответ занимает около 4 секунд, редкие до 7. Уточнить агент спросил на 8 неоднозначных вопросах из 10 и ещё на 4 из 40 обычных. На золотом наборе он уточнял 3 раза из 50, там вопросы конкретные.

### Цепочки рассуждений

10 примеров из val: 3 неоднозначных и 7 многошаговых. Мысль это текст, который агент пишет перед вызовом, дальше какой инструмент, с каким запросом и что он нашёл.

In [21]:
def show_trace(x, r):
    print(f"{x['aid']} {x['kind']} | {x['question']}")
    prev = None
    for n, t in enumerate(r['trace'], 1):
        posts = ', '.join(dict.fromkeys(f"{chunk_doc[i]} ({str(chunks[i]['date'])[:10]})" for i in t['idx']))
        if t['thought'] and t['thought'] != prev:
            print(f'   мысль: {t["thought"][:220]}')
        prev = t['thought']
        print(f'   шаг {n}, {t["tool"]}("{t["arg"]}"):', ('ответ пользователя: ' + t['result'][:120]) if t['tool'] == 'ask_user' else ('нашёл ' + posts[:230]))
    print('   ответ:', r['answer'][:350].replace('\n', ' '))
    print('   нужные посты:', ', '.join(x['doc_ids']), '| найдено из них:', sum(d in {chunk_doc[i] for i in r['context']} for d in x['doc_ids']))
    print()

val_agent = [x for x in agent_set if x['split'] == 'val']
for x in [x for x in val_agent if x['clarification']][:3] + [x for x in val_agent if not x['clarification']][:7]:
    show_trace(x, AGENT[x['aid']])

a04 неоднозначный | Что за ремейк может выйти совсем скоро?
   мысль: Уточню, о каком ремейке речь — в постах их может быть несколько.
   шаг 1, ask_user("О каком ремейке речь? Уточните игру, серию или компанию — или вас интересует любой анонс ремейка со скорым релизом?"): ответ пользователя: Я про ту новость, где из Steam удалили трейлеры
   мысль: Ищу новость про удалённые из Steam трейлеры и связанный с ней ремейк.
   шаг 2, search_bm25("Steam удалили трейлеры ремейк"): нашёл egs_tg/9933 (2026-01-31), SteamByFree/17828 (2026-07-04), SteamByFree/17985 (2026-07-11), egs_tg/10071 (2026-03-05), egs_tg/10867 (2026-08-26)
   шаг 3, search_vector("из Steam удалили трейлеры игры, скорый релиз ремейка"): нашёл egs_tg/9933 (2026-01-31), egs_tg/9786 (2025-12-29), egs_tg/10175 (2026-04-02), SteamByFree/17723 (2026-06-30)
   ответ: Ремейк Fallout 3. Из Steam убрали трейлеры всех версий Fallout 3, а такие «чистки» обычно происходят перед крупными анонсами или релизами. Похожая ситуация была с Obl

Видно, как агент выбирает инструменты. Для конкретных названий и цифр (Assassin's Creed Black Flag Resynced, PIF, 55 млрд) он идёт в BM25, для описаний своими словами в векторный. Когда первый поиск находит кандидата, он проверяет его отдельным запросом: в a01 нашёл Rockstar и отдельно искал взлом с выручкой. В a04 и a16 он сразу спросил, о чём речь, а в a11 сначала поискал по слову иск, увидел семь разных исков и уже потом переспросил, перечислив варианты. В a07 агент нашёл оба нужных поста, но не понял вопрос: ответ там Windows 10, а он решил, что ОС не названа. Так что ошибки бывают и после удачного поиска.

### Сравнение с обычным RAG

Baseline тут обычный RAG: hybrid топ-5 и тот же промпт генерации. Реранкер из КТ4 для новых вопросов не беру, он работал на Claude. Для неоднозначных вопросов обычный RAG получает только сам вопрос, переспросить он не может. Судья везде видит вопрос вместе с уточнением, то есть оценивает по тому, что пользователь на самом деле хотел. Faithfulness и utilization у агента считаются по всем фрагментам, которые он нашёл за все поиски. Нужных постов в контексте это доля постов, по которым составлен вопрос, найденных системой. У агента фрагментов в контексте больше, чем 5, так что по этому столбцу сравнение заранее в его пользу.

In [22]:
def full_question(x):
    return x['question'] + (f" (уточнение пользователя: {x['clarification']})" if x['clarification'] else '')

AB = {x['aid']: list(map(int, hybrid(x['question'])[:5])) for x in agent_set}
run_parallel(lambda x: generate(x['question'], AB[x['aid']]), agent_set)
SYSTEMS = {
    'обычный RAG (hybrid, топ-5)': {x['aid']: (generate(x['question'], AB[x['aid']])['text'], AB[x['aid']]) for x in agent_set},
    'агент': {x['aid']: (AGENT[x['aid']]['answer'], AGENT[x['aid']]['context']) for x in agent_set},
}
jobs = [(sname, x, c) for sname in SYSTEMS for x in agent_set for c in CRITERIA]
run_parallel(lambda j: judge(j[2], full_question(j[1]), SYSTEMS[j[0]][j[1]['aid']][1], SYSTEMS[j[0]][j[1]['aid']][0]), jobs)
ASCORE = {(sname, x['aid'], c): judge(c, full_question(x), SYSTEMS[sname][x['aid']][1], SYSTEMS[sname][x['aid']][0])['score'] for sname, x, c in jobs}

def found_share(sname, x):
    posts = {chunk_doc[i] for i in SYSTEMS[sname][x['aid']][1]}
    return np.mean([d in posts for d in x['doc_ids']])

rows = []
for sname in SYSTEMS:
    for split in ['val', 'test']:
        xs = [x for x in agent_set if x['split'] == split]
        rows.append({'система': sname, 'split': split, **{c: round(np.mean([ASCORE[sname, x['aid'], c] for x in xs]), 2) for c in CRITERIA},
                     'нужных постов в контексте': round(np.mean([found_share(sname, x) for x in xs]), 2),
                     'отказов': sum(refused(SYSTEMS[sname][x['aid']][0]) for x in xs)})
pd.DataFrame(rows)

,система,split,faithfulness,relevance,utilization,нужных постов в контексте,отказов
0,"обычный RAG (hybrid, топ-5)",val,5.00,3.17,4.37,0.74,8
1,"обычный RAG (hybrid, топ-5)",test,4.87,3.60,3.87,0.77,2
2,агент,val,4.94,4.57,4.83,0.93,2
3,агент,test,5.00,4.87,4.60,0.97,0


In [23]:
rows = []
for kind in ['сводка', 'развитие событий', 'сравнение', 'причина и следствие', 'неоднозначный']:
    xs = [x for x in agent_set if x['kind'] == kind]
    rows.append({'тип': kind, 'вопросов': len(xs), **{f'{c[:5]}, {s_[:5]}': round(np.mean([ASCORE[s_, x['aid'], c] for x in xs]), 2) for s_ in SYSTEMS for c in ['relevance', 'faithfulness']},
                 'нужных постов, RAG': round(np.mean([found_share('обычный RAG (hybrid, топ-5)', x) for x in xs]), 2),
                 'нужных постов, агент': round(np.mean([found_share('агент', x) for x in xs]), 2)})
pd.DataFrame(rows)

,тип,вопросов,"relev, обычн","faith, обычн","relev, агент","faith, агент","нужных постов, RAG","нужных постов, агент"
0,сводка,18,3.89,5.0,4.83,4.89,0.81,0.95
1,развитие событий,15,3.00,5.0,4.60,5.00,0.79,0.97
2,сравнение,5,2.80,5.0,3.80,5.00,0.70,1.00
3,причина и следствие,2,2.00,5.0,4.00,5.00,0.75,0.75
4,неоднозначный,10,3.20,4.8,5.00,5.00,0.60,0.90


Агент выигрывает по relevance и на val, и на test, больше чем на балл. Нужные посты он собирает в контекст почти всегда, а обычный RAG примерно в трёх случаях из четырёх. Отказов у обычного RAG 10, у агента 2. Faithfulness у обоих около 5.

По типам больше всего агент выигрывает на неоднозначных, потому что уточняет, и на вопросах про развитие событий, где нужны посты за разные даты. На сводках прирост меньше, там и обычный RAG справлялся неплохо. Хуже всего у агента сравнения, 3.8: нужные посты он находит, но не всегда правильно их сопоставляет. Причин и следствий в наборе всего 2 вопроса, по ним выводов не делаю.

Прогоняю судью и по ответам агента на золотом наборе, чтобы поставить агента в сводную таблицу.

In [24]:
AGENT_NAME = 'КТ5: агент'
run_parallel(lambda j: judge(j[1], j[0]['question'], AGENT_GOLD[j[0]['qid']]['context'], AGENT_GOLD[j[0]['qid']]['answer']), [(g, c) for g in answerable for c in CRITERIA])
for g in gold:
    ANS[AGENT_NAME, g['qid']] = {'text': AGENT_GOLD[g['qid']]['answer']}
    if g['type'] != 'negative':
        for c in CRITERIA:
            SCORE[AGENT_NAME, g['qid'], c] = judge(c, g['question'], AGENT_GOLD[g['qid']]['context'], AGENT_GOLD[g['qid']]['answer'])['score']
CONTEXT = {(name, g['qid']): TOP5[name][g['qid']] for name in RETRIEVERS for g in gold} | {(AGENT_NAME, g['qid']): AGENT_GOLD[g['qid']]['context'] for g in gold}
print('агент на золотом наборе:', gen_metrics(AGENT_NAME, val_q), '| test:', gen_metrics(AGENT_NAME, test_q))

агент на золотом наборе: {'faithfulness': 4.82, 'relevance': 4.61, 'utilization': 4.68, 'отказы на негативных': '7 из 7', 'ложные отказы': '1 из 28'} | test: {'faithfulness': 4.92, 'relevance': 4.83, 'utilization': 4.92, 'отказы на негативных': '2 из 3', 'ложные отказы': '0 из 12'}


### Ручная проверка судьи

Беру 20 случайных вопросов из val и по каждому случайно ответ одной из трёх систем: baseline, с реранкером или агент. Сам оцениваю их по тем же трём критериям и той же шкале, оценки судьи при этом не видел. Потом считаю взвешенную Cohen's Kappa между мной и судьёй по каждому критерию.

In [25]:
rng = random.Random(7)
val_pos = [g for g in val_q if g['type'] != 'negative']
HUMAN_ITEMS = [{'id': f'h{i + 1:02d}', 'qid': g['qid'], 'config': rng.choice([BASE, BEST, AGENT_NAME])} for i, g in enumerate(rng.sample(val_pos, 20))]
print('вопросов в выборке:', len(HUMAN_ITEMS), 'из', len(val_pos), 'вопросов с ответом в val')
pd.DataFrame(HUMAN_ITEMS).groupby('config').size()

вопросов в выборке: 20 из 28 вопросов с ответом в val


config
КТ3: hybrid RRF        8
КТ4: + LLM-реранкер    4
КТ5: агент             8
dtype: int64

In [26]:
human = {x['id']: x for x in json.load(open(DATA / 'kt5_human_ratings.json', encoding='utf-8'))}
rows, pairs_h = [], []
for c in CRITERIA:
    h = np.array([human[it['id']][c] for it in HUMAN_ITEMS])
    j = np.array([SCORE[it['config'], it['qid'], c] for it in HUMAN_ITEMS])
    pairs_h += [{'id': it['id'], 'qid': it['qid'], 'критерий': c, 'человек': a, 'судья': b} for it, a, b in zip(HUMAN_ITEMS, h, j)]
    rows.append({'критерий': c, 'kappa, квадратичные веса': round(cohen_kappa_score(h, j, weights='quadratic', labels=[1, 2, 3, 4, 5]), 2),
                 'kappa, линейные веса': round(cohen_kappa_score(h, j, weights='linear', labels=[1, 2, 3, 4, 5]), 2),
                 'точное совпадение': f'{100 * np.mean(h == j):.0f}%', 'разница не больше 1': f'{100 * np.mean(abs(h - j) <= 1):.0f}%',
                 'среднее, человек': round(h.mean(), 2), 'среднее, судья': round(j.mean(), 2)})
ph = pd.DataFrame(pairs_h)
print('расхождения на 2 балла и больше:')
print(ph[abs(ph['человек'] - ph['судья']) >= 2].to_string(index=False))
pd.DataFrame(rows)

расхождения на 2 балла и больше:
 id qid    критерий  человек  судья
h16 q22   relevance        5      3
h06 q03 utilization        5      3
h11 q33 utilization        3      5
h18 q18 utilization        3      5
h20 q40 utilization        3      5


,критерий,"kappa, квадратичные веса","kappa, линейные веса",точное совпадение,разница не больше 1,"среднее, человек","среднее, судья"
0,faithfulness,0.64,0.64,95%,100%,4.9,4.95
1,relevance,0.89,0.78,75%,95%,4.1,3.90
2,utilization,0.00,-0.02,60%,80%,4.5,4.80


По faithfulness и relevance судья надёжен, kappa выше 0.6. По faithfulness почти все оценки 5 и у меня, и у судьи, разошлись мы только на одном ответе агента (h09), где я нашёл мелкую неточность, а судья нет. Так что kappa тут держится на одном-двух ответах, но выдумок, которые судья пропустил бы, я не нашёл. По relevance расхождения в основном на один балл, сильно разошлись только на h16: я считаю, что на вопрос ответили, а судья поставил 3.

По utilization kappa 0, судья ненадёжен. Расхождения в обе стороны. Чаще судья мягче меня: в q18 ответ пишет, что про внутренние студии во фрагментах ничего не сказано, а в пятом фрагменте прямо написано, что отказ от дисков касается и внутренних студий. Судья верит ответу на слово и ставит 5, я ставлю 3. Так же в q40 и на длинных ответах агента (q33, q37). А в q03 наоборот: судья снизил за фрагмент, который просто по той же теме, но на вопрос не отвечает.

In [27]:
JUDGE['v5'] = {**JUDGE['v2'], 'utilization': JUDGE_HEAD + '''Оцени context utilization: использует ли ответ всю информацию из контекста, которая относится к вопросу, или только её часть.
Правила: сначала сам найди во фрагментах всё, что прямо отвечает на вопрос или на какую-то его часть. Фрагменты, которые только по той же теме, но на вопрос не отвечают, не нужны, и если их нет в ответе, оценку не снижай. Не верь ответу на слово: если в ответе написано, что какой-то информации во фрагментах нет, проверь это сам, и если она там есть, считай её упущенной. Если нужной информации в контексте нет и ответ это говорит, это 5.
5 - ответ использует всё, что во фрагментах прямо отвечает на вопрос
4 - упущена одна мелкая деталь
3 - упущен один нужный факт или фрагмент, в том числе если ответ ошибочно пишет, что его нет
2 - использована малая часть нужной информации
1 - нужная информация в контексте есть, но ответ её не использует или говорит «Не знаю»''' + JUDGE_TAIL2}

JUDGE['v6'] = {**JUDGE['v2'], 'utilization': JUDGE_HEAD + '''Оцени context utilization: использует ли ответ всю информацию из контекста, которая нужна для ответа на вопрос.
Разбор: выпиши части вопроса. Для каждой части найди во фрагментах прямой ответ на неё и укажи номер фрагмента, или напиши, что ответа нет. Фрагменты, которые только по той же теме и на вопрос не отвечают, не считай. Потом проверь, взял ли ответ системы то, что ты нашёл. Если ответ пишет, что информации нет, а ты её нашёл, это упущение.
5 - ответ взял всё, что ты нашёл, или ты ничего не нашёл и ответ это говорит
4 - взято всё, но одна мелкая деталь упущена
3 - упущен ответ на одну часть вопроса или один нужный фрагмент
2 - упущено больше половины найденного
1 - ответ во фрагментах есть, но система его не использует или говорит «Не знаю»''' + JUDGE_TAIL3}

def agreement(version, ps, criterion):
    run_parallel(lambda j: judge(criterion, j[0]['q'], j[0]['idx'], j[0]['a'], version, j[1]), [(p, run) for p in ps for run in (1, 2)])
    return np.mean([judge(criterion, p['q'], p['idx'], p['a'], version, 1)['score'] == judge(criterion, p['q'], p['idx'], p['a'], version, 2)['score'] for p in ps])

items_h = [(it, it['qid'], CONTEXT[it['config'], it['qid']], ANS[it['config'], it['qid']]['text']) for it in HUMAN_ITEMS]
qtext = {g['qid']: g['question'] for g in gold}
h = np.array([human[it['id']]['utilization'] for it in HUMAN_ITEMS])
util = {'человек': h, 'v2': np.array([SCORE[it['config'], it['qid'], 'utilization'] for it in HUMAN_ITEMS])}
rows = []
for version in ['v5', 'v6']:
    run_parallel(lambda x: judge('utilization', qtext[x[1]], x[2], x[3], version), items_h)
    util[version] = np.array([judge('utilization', qtext[q], idx, a, version)['score'] for it, q, idx, a in items_h])
    rows.append({'версия': version, 'agreement, первые 30 пар': f"{100 * agreement(version, pairs, 'utilization'):.0f}%",
                 'agreement, другие 30 пар': f"{100 * agreement(version, holdout, 'utilization'):.0f}%",
                 'kappa с человеком': round(cohen_kappa_score(h, util[version], weights='quadratic', labels=[1, 2, 3, 4, 5]), 2),
                 'точное совпадение': f'{100 * np.mean(h == util[version]):.0f}%', 'разница не больше 1': f'{100 * np.mean(abs(h - util[version]) <= 1):.0f}%'})
diff = pd.DataFrame({'id': [it['id'] for it in HUMAN_ITEMS], 'qid': [it['qid'] for it in HUMAN_ITEMS], **util})
print(diff[(diff['человек'] != diff['v2']) | (diff['человек'] != diff['v5']) | (diff['человек'] != diff['v6'])].to_string(index=False))
pd.DataFrame(rows)

 id qid  человек  v2  v5  v6
h05 q02        5   4   5   4
h06 q03        5   3   5   5
h09 q16        4   5   5   4
h11 q33        3   5   3   5
h15 q31        4   5   5   5
h17 q37        3   4   5   5
h18 q18        3   5   5   5
h20 q40        3   5   5   3


,версия,"agreement, первые 30 пар","agreement, другие 30 пар",kappa с человеком,точное совпадение,разница не больше 1
0,v5,93%,100%,0.3,75%,85%
1,v6,90%,87%,0.3,75%,85%


По заданию при kappa ниже 0.4 промпт надо переделать и повторить. Faithfulness и relevance оставляю как в v2, переделываю только utilization:
- **v5**: добавил два правила под эти расхождения. Нужным считается только то, что прямо отвечает на вопрос или его часть, фрагменты просто по теме не нужны. И не верить ответу на слово: если ответ пишет, что информации нет, проверить самому. q03 и q33 сошлись с моими оценками, kappa стала 0.3.
- **v6**: попробовал заставить судью сначала самому найти во фрагментах ответ на каждую часть вопроса, а потом сравнить с ответом системы. Kappa та же, а стабильность хуже.

Оставляю v5: она стабильна, и kappa у неё выше, чем у v2. Но до 0.4 она не дотянула, и q18 не видит ни одна версия. Проверял я на тех же 20 ответах, по которым правил, так что даже 0.3 скорее завышено. Итог: по faithfulness и relevance судье можно доверять, по utilization нет. Поэтому все выводы дальше я делаю по faithfulness и relevance, а utilization в таблицах выше оставил для справки.

## 5. Сводный отчёт

### Сводная таблица по всем КТ

Строки это ключевые конфигурации от КТ3 до агента из КТ5, все с одной и той же генерацией. Recall@5 и NDCG@5 это поиск, faithfulness и relevance это судья на вопросах с ответом. Задержка это полный путь от вопроса до ответа, по всем 50 вопросам золотого набора. Время поиска BM25 и Chroma мерил сейчас, эмбеддинг запроса и cross-encoder взял из замеров КТ4, перефразировки и реранкер тоже из КТ4, там это был Claude. Генерация и агент это время ответов DeepSeek, а запросы к нему шли в несколько потоков, так что это время может быть немного завышено. У агента Recall@5 и NDCG@5 нет: он не выдаёт один топ-5, а ищет несколько раз.

In [28]:
def live_split_times():
    out = {}
    for g in gold:
        t0 = time.perf_counter(); bm25_list(g['question']); t1 = time.perf_counter(); chroma_query(g['question'], DEPTH); t2 = time.perf_counter()
        out[g['question']] = {'bm25': 1000 * (t1 - t0), 'chroma': 1000 * (t2 - t1)}
    return out

SPLIT_T = measured('kt5_bm25_chroma', live_split_times)
LIVE = TIMES['latency_live']

def latency_ms(name, g):
    q, t = g['question'], LIVE[g['question']]
    gen = 1000 * ANS[name, g['qid']].get('sec', 0)
    if name == AGENT_NAME:
        return 1000 * AGENT_GOLD[g['qid']]['sec']
    ret = {'КТ3: BM25': SPLIT_T[q]['bm25'], 'КТ3: dense FRIDA': t['emb'] + SPLIT_T[q]['chroma'], 'КТ3: hybrid RRF': t['emb'] + t['search'],
           'КТ4: + multi-query': 1000 * kt4_llm('mq', MQ_PROMPT.replace('{q}', q))['sec'] + t['emb_mq'] + t['search_mq'],
           'КТ4: + cross-encoder': t['emb'] + t['search'] + t['ce'], 'КТ4: + LLM-реранкер': t['emb'] + t['search'] + 1000 * lr_call(q, hybrid(q)[:LR_N])['sec']}[name]
    return ret + gen

rows = []
for name in list(RETRIEVERS) + [AGENT_NAME]:
    row = {'конфигурация': name}
    for split, qs in [('val', val_q), ('test', test_q)]:
        pos = [g for g in qs if g['type'] != 'negative']
        ret = summary(RET[name][0 if split == 'val' else 1]) if name in RET else None
        row[f'{split} R@5'] = ret['R@5'] if ret is not None else '-'
        row[f'{split} NDCG@5'] = ret['NDCG@5'] if ret is not None else '-'
        row[f'{split} faithfulness'] = round(np.mean([SCORE[name, g['qid'], 'faithfulness'] for g in pos]), 2)
        row[f'{split} relevance'] = round(np.mean([SCORE[name, g['qid'], 'relevance'] for g in pos]), 2)
    lat = np.array([latency_ms(name, g) for g in gold]) / 1000
    row['медиана, с'], row['p95, с'] = round(np.median(lat), 2), round(np.percentile(lat, 95), 2)
    row['отказы на негативных'] = sum(refused(ANS[name, g['qid']]['text']) for g in gold if g['type'] == 'negative')
    rows.append(row)
summary_table = pd.DataFrame(rows)
summary_table

,конфигурация,val R@5,val NDCG@5,val faithfulness,val relevance,test R@5,test NDCG@5,test faithfulness,test relevance,"медиана, с","p95, с",отказы на негативных
0,КТ3: BM25,0.737,0.719,4.93,3.61,0.75,0.655,5.00,4.00,1.15,1.66,10
1,КТ3: dense FRIDA,0.699,0.675,5.00,3.82,0.729,0.73,5.00,4.17,1.12,1.82,10
2,КТ3: hybrid RRF,0.743,0.713,4.93,3.71,0.771,0.759,5.00,4.17,1.10,1.64,10
3,КТ4: + multi-query,0.698,0.655,5.00,3.86,0.771,0.709,5.00,4.08,3.40,4.59,9
4,КТ4: + cross-encoder,0.729,0.719,4.86,3.89,0.792,0.681,4.92,4.17,1.17,1.75,8
5,КТ4: + LLM-реранкер,0.763,0.734,5.00,3.86,0.792,0.761,5.00,4.42,2.63,3.83,10
6,КТ5: агент,-,-,4.82,4.61,-,-,4.92,4.83,3.04,6.96,9


По поиску всё двигалось мало: от BM25 до реранкера Recall@5 на val вырос на 0.03, и почти так же мало двигалась relevance. У multi-query и cross-encoder поиск не стал лучше, relevance на val чуть выросла, на test нет. Всё это в пределах шума: на val всего 28 вопросов с ответом. Faithfulness везде около 5, модель почти никогда не выдумывает при любом поиске. Сильно сдвинул качество только агент: relevance на val выросла с 3.86 до 4.61. Faithfulness у него чуть ниже: в длинных сводках по многим постам он иногда добавляет мелкую неточность, например называет игры новинками, хотя в постах этого нет. По задержке агент стоит примерно как реранкер, около 3 секунд, но хвост у него длиннее. В обычных конфигурациях почти всё время уходит на генерацию, около секунды.

На негативных все конфигурации отказываются почти всегда. Там, где в столбце не 10, модель тоже ничего не выдумала, просто написала другими словами, что во фрагментах этого нет.

Финальная система КТ5: агент.

### Где система теряет качество

Для лучшей обычной конфигурации (с реранкером) и для агента разбиваю вопросы с ответом на три группы: поиск нашёл все нужные посты из разметки, часть или ни одного. И смотрю, хороший получился ответ (relevance 4-5) или плохой. Ниже relevance по типам вопросов, val и test вместе.

In [29]:
def retrieval_state(name, g):
    rel = {dup_map.get(d, d) for d in g['relevant_doc_ids']}
    if name == AGENT_NAME:
        got = {chunk_doc[i] for i in AGENT_GOLD[g['qid']]['context']}
    else:
        got = set(chunk_doc[TOP5[name][g['qid']]])
    share = len(rel & got) / len(rel)
    return 'нашёл все нужные посты' if share == 1 else 'нашёл часть' if share > 0 else 'не нашёл ни одного'

rows = []
for name in [BEST, AGENT_NAME]:
    for g in answerable:
        rows.append({'система': name, 'qid': g['qid'], 'тип': g['type'], 'поиск': retrieval_state(name, g), 'relevance': SCORE[name, g['qid'], 'relevance']})
bn = pd.DataFrame(rows)
bn['ответ'] = np.where(bn['relevance'] >= 4, 'хороший (4-5)', 'плохой (1-3)')
print(pd.crosstab([bn['система'], bn['поиск']], bn['ответ']))
print()
pd.pivot_table(bn, index='тип', columns='система', values='relevance', aggfunc='mean').round(2)

ответ                                       плохой (1-3)  хороший (4-5)
система             поиск                                              
КТ4: + LLM-реранкер нашёл все нужные посты             6             24
                    нашёл часть                        2              1
                    не нашёл ни одного                 6              1
КТ5: агент          нашёл все нужные посты             4             26
                    нашёл часть                        0              3
                    не нашёл ни одного                 0              7



система,КТ4: + LLM-реранкер,КТ5: агент
тип,,
abstract,2.70,4.80
composite,4.07,4.40
factual,4.87,4.87


**Где система теряет больше: на поиске или на генерации.** Зависит от типа вопроса. На фактических и составных поиск находит все нужные посты из разметки всегда, у обеих систем. Там качество теряется на генерации: у обычного RAG 6 плохих ответов при найденных постах, и в 5 из них в контекст попал чанк, а нужный факт был в другой части того же поста. В q01 в чанке одна игра из трёх, в q28 от поста про раздачу в чанк попала одна строка без названий игр, в q25 цены в длинной таблице. На абстрактных наоборот теряется на поиске: посты из разметки не находятся почти никогда, ни одним поиском. Агент находит их не лучше, просто собирает ответ из других постов по теме, и судья ставит ему 4-5. Но relevance проверяет, есть ли ответ, а не верен ли он, так что на абстрактных оценка агента скорее завышена. Итого у обычного RAG 14 плохих ответов из 40: 8 на абстрактных из-за поиска и 6 на остальных из-за обрезанного контекста. У агента плохих 4, и все при найденных постах.

**Какие вопросы так и не получились.** Абстрактные, вроде какие жанры чаще всего в подборках или что будет с индустрией. Поиск с одним запросом их не тянет ни в одной КТ: Recall@5 около нуля. Агент отвечает на них примерами из нескольких постов, но настоящей статистики по корпусу такой поиск не даст. Из остальных у агента слабее всего составные вопросы, 4.4.

**Что дало больше всего за свои усилия.** Агент: один цикл с тремя инструментами и копейки на вопрос подняли relevance на val с 3.86 до 4.61. Всё, что я делал с поиском в КТ3 и КТ4 (подбор параметров BM25, hybrid, реранкеры), сдвинуло Recall@5 на 0.02-0.03. Из дешёвого лучше всего сработала чистка HTML в КТ2: корпус ужался больше чем вдвое, а поиск стал лучше.

### Выводы по КТ

**КТ1.** Собрал 1650 постов из двух Telegram-каналов про игры. Источник правда грязный: больше половины символов это HTML, плюс подписи канала, эмодзи и дубли. Золотой набор из 50 вопросов генерировал LLM, а я проверял руками и правил. Сработало разделение на четыре типа вопросов, по нему потом в каждой КТ было видно, где проблемы. Неожиданно было, что модель без контекста почти ничего не знает про эти новости: ни одного правильного ответа из 28. И что судья тогда помечал отказы то так, то эдак, поэтому в КТ5 правила для отказов прописаны в промпте явно.

**КТ2.** Сработала чистка HTML и служебки: корпус ужался больше чем вдвое, поиск стал лучше. Лучшим чанкингом оказались предложения по 128 токенов, а метаданные в начале чанка немного помогли. Не сработали обработка чанков через LLM (дорого и без прироста) и фильтры по типу поста. Неожиданно было, что parent/child нужен не для поиска, а для генерации: целый пост содержит факт целиком. В КТ5 это подтвердилось.

**КТ3.** FRIDA оказалась лучшей моделью эмбеддингов, но настроенный BM25 по NDCG@5 на val был даже лучше неё, на test наоборот. Гибрид через RRF по Recall@5 чуть лучше обоих. Порядок моделей совпал с MTEB, а вот разница между ними на корпусе оказалась в разы меньше, чем в бенчмарке. Для абстрактных вопросов не сработало ничего: Recall@5 около нуля у всех.

**КТ4.** Multi-query сделал хуже, cross-encoder почти ничего не дал, а английский вообще развалился на русском. Сработал только LLM-реранкер, и то на 0.02 по Recall@5, и стабильным он стал только с третьей версией промпта, где модель ставит оценки, а сортирую я сам. Неожиданно было, что чем больше пул для cross-encoder, тем хуже.

**КТ5.** Генерация почти не выдумывает и на негативных вопросах честно говорит Не знаю. Судья стабилен, и по faithfulness и relevance он ставит почти как я, а вот utilization у него с моими оценками не сходится даже после двух переделок. Попытки сделать судью строже добавлением правил только ухудшали стабильность. Агент дал самый большой прирост за весь проект, особенно на неоднозначных и многошаговых вопросах. Неожиданно было, сколько качества теряется уже после поиска: нужный пост найден, но в контекст попал не тот его кусок.

### Что делать дальше

1. **Отдавать в генерацию весь пост вместо чанка.** Это первое, потому что дёшево и закрывает 5 из 6 плохих ответов на фактических и составных вопросах, где поиск отработал правильно. Посты в корпусе короткие, медиана около 300 символов, так что контекст вырастет не сильно. В КТ2 parent/child уже показал, что факт в целом посте находится заметно чаще.
2. **Роутинг между агентом и обычным RAG.** На фактических вопросах агент не лучше обычного RAG, relevance у обоих одинаковая, а отвечает он почти в 3 раза дольше и с хвостом до 7 секунд. Простые вопросы с конкретным названием можно отдавать обычному RAG, а агенту только составные, абстрактные и неоднозначные.
3. **Переделать судью через эталон из золотого набора.** Тут две проблемы. Utilization не сходится с человеком (kappa 0.3 даже после двух переделок): судья сам решает, какая информация нужна, и ошибается. А relevance не проверяет, верен ли ответ, поэтому на абстрактных вопросах агент получает высокие оценки за ответ по другим постам. В золотом наборе у каждого вопроса уже есть эталонный ответ и цитаты из постов, так что судья может проверять ответ по ним, а не угадывать, что нужно.